In [1119]:
#librerias
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.stats import norm
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import os

In [1120]:
df_final=pd.read_csv('../results/fechas_117/df_20250401_to_20250909_M20_preproc.csv')

In [1121]:
df_final['Fecha'].unique()

<StringArray>
['2025-05-25', '2025-04-07', '2025-07-01', '2025-07-02', '2025-04-08',
 '2025-08-15', '2025-09-03', '2025-08-28', '2025-06-12', '2025-06-09',
 ...
 '2025-05-14', '2025-08-25', '2025-06-24', '2025-04-10', '2025-05-15',
 '2025-05-01', '2025-04-09', '2025-06-17', '2025-08-12', '2025-09-05']
Length: 117, dtype: str

## Instrumentos

In [1122]:
# Fechas referencia
fechas_ref = pd.Index(sorted(pd.to_datetime(df_final['Fecha'].unique())))

def fill_fechas(df):
    return (df.assign(Fecha=pd.to_datetime(df['Fecha']))
              .set_index('Fecha')
              .reindex(fechas_ref)
              .ffill()
              .bfill()
              .rename_axis('Fecha')
              .reset_index())



In [1123]:
df_instrumentos=pd.read_csv('../datos/procesado/instrumentos.csv')

archivo_rios = "../datos/raw/rios.xlsx"
df_rios = pd.read_excel(archivo_rios, header=3)
df_rios = df_rios[['Fecha', 'Aportes %', 'Nombre Río']]
df_rios = df_rios[df_rios["Nombre Río"] != "FLORIDA II"].copy()
df_rios_agg = (
    df_rios.groupby('Fecha')['Aportes %']
           .apply(lambda x: np.mean(x))   # promedio por fecha
           .reset_index()
           .rename(columns={'Aportes %': 'aportes_rios'})
)
# Aplicar a ambos
df_rios_agg = fill_fechas(df_rios_agg)
df_instrumentos = fill_fechas(df_instrumentos)

df_instrumentos=pd.merge(df_instrumentos,df_rios_agg, on='Fecha' )

In [1124]:
df_instrumentos['month']= pd.to_datetime(df_instrumentos['Fecha']).dt.month
month_dummies = pd.get_dummies(df_instrumentos['month'], prefix='month')
day_dummies = pd.get_dummies(df_instrumentos['weekday'], prefix='day')
df_instrumentos = pd.concat([df_instrumentos, month_dummies, day_dummies], axis=1)

In [1125]:


data = {
    "Mes": ["2025-04", "2025-05", "2025-06", "2025-07", "2025-08", "2025-09"],
    "prob_nino": [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
}

nino = pd.DataFrame(data)
nino['month']=pd.to_datetime(nino['Mes']).dt.month
df_instrumentos=pd.merge(df_instrumentos, nino[['month', 'prob_nino']], on='month', how='left' )

In [1126]:
import pandas as pd
import ast
df_input_red = pd.read_csv(
    "../results/fechas_117/TERMICA/inputRed.csv",
    sep=";",
    decimal=","
)

# Asegurar que la columna sea lista real (por si viene como string)
df_input_red['daily_reman_demand'] = df_input_red['daily_reman_demand'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Expandir el array en columnas (24 horas)
df_demand_thermic = pd.DataFrame(
    df_input_red['daily_reman_demand'].tolist(),
    columns=[f"demanda_h{h:02d}" for h in range(24)]
) #/1000

# Agregar la fecha
df_demand_thermic['Fecha'] = df_input_red['Fecha']

# Opcional: ordenar columnas
df_demand_thermic = df_demand_thermic[['Fecha'] + [f"demanda_h{h:02d}" for h in range(24)]]

In [1127]:
df_instrumentos['Fecha']=pd.to_datetime(df_instrumentos['Fecha'])
df_demand_thermic['Fecha']=pd.to_datetime(df_demand_thermic['Fecha'])
df_instrumentos=pd.merge(df_instrumentos,df_demand_thermic, on='Fecha' )

In [1128]:
#df_instrumentos = df_instrumentos[df_instrumentos['weekday'] != 'Sunday']

## Parametrizacion

In [1129]:
df_z1=pd.read_csv('../datos/procesado/parametrizacion.csv')
df_z1['Fecha']=pd.to_datetime(df_z1['Fecha'])
df_z1=pd.merge(df_z1,df_instrumentos, on='Fecha')

In [1130]:
df_final['Fecha'].unique()

<StringArray>
['2025-05-25', '2025-04-07', '2025-07-01', '2025-07-02', '2025-04-08',
 '2025-08-15', '2025-09-03', '2025-08-28', '2025-06-12', '2025-06-09',
 ...
 '2025-05-14', '2025-08-25', '2025-06-24', '2025-04-10', '2025-05-15',
 '2025-05-01', '2025-04-09', '2025-06-17', '2025-08-12', '2025-09-05']
Length: 117, dtype: str

In [1131]:
df_instrumentos['Fecha'].unique()

<DatetimeArray>
['2025-04-01 00:00:00', '2025-04-02 00:00:00', '2025-04-03 00:00:00',
 '2025-04-04 00:00:00', '2025-04-07 00:00:00', '2025-04-08 00:00:00',
 '2025-04-09 00:00:00', '2025-04-10 00:00:00', '2025-04-11 00:00:00',
 '2025-04-14 00:00:00',
 ...
 '2025-08-28 00:00:00', '2025-08-29 00:00:00', '2025-08-31 00:00:00',
 '2025-09-01 00:00:00', '2025-09-02 00:00:00', '2025-09-03 00:00:00',
 '2025-09-04 00:00:00', '2025-09-05 00:00:00', '2025-09-08 00:00:00',
 '2025-09-09 00:00:00']
Length: 117, dtype: datetime64[us]

In [1132]:
df_z1['Fecha'].unique()

<DatetimeArray>
['2025-04-01 00:00:00', '2025-04-02 00:00:00', '2025-04-03 00:00:00',
 '2025-04-04 00:00:00', '2025-04-07 00:00:00', '2025-04-08 00:00:00',
 '2025-04-09 00:00:00', '2025-04-10 00:00:00', '2025-04-11 00:00:00',
 '2025-04-14 00:00:00',
 ...
 '2025-08-28 00:00:00', '2025-08-29 00:00:00', '2025-08-31 00:00:00',
 '2025-09-01 00:00:00', '2025-09-02 00:00:00', '2025-09-03 00:00:00',
 '2025-09-04 00:00:00', '2025-09-05 00:00:00', '2025-09-08 00:00:00',
 '2025-09-09 00:00:00']
Length: 117, dtype: datetime64[us]

In [1133]:
df_final['Fecha']=pd.to_datetime(df_final['Fecha'])
df_final_instrumentos=pd.merge(df_final, df_z1[['CodigoPlanta', 'Fecha','id_planta', 'Combustible por Defecto',
       'Nombre Recurso', 'Agente Representante', 'COMBUSTOLEO', 'CERE',
       'FAZNI', 'trm', 'pfuel','petroleo', 'nivel_embalses', 'weekday', 'aportes_rios',
       'month', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8',
       'month_9', 'day_Friday', 'day_Monday', 'day_Sunday', 'day_Thursday',
       'day_Tuesday', 'day_Wednesday', 'demanda_h00', 'demanda_h01',
       'demanda_h02', 'demanda_h03', 'demanda_h04', 'demanda_h05',
       'demanda_h06', 'demanda_h07', 'demanda_h08', 'demanda_h09',
       'demanda_h10', 'demanda_h11', 'demanda_h12', 'demanda_h13',
       'demanda_h14', 'demanda_h15', 'demanda_h16', 'demanda_h17',
       'demanda_h18', 'demanda_h19', 'demanda_h20', 'demanda_h21',
       'demanda_h22', 'demanda_h23', 'prob_nino' ]], on=['CodigoPlanta', 'Fecha'], how='left')

In [1134]:
df_final_instrumentos['Hora']=pd.to_datetime(df_final_instrumentos['FechaHora']).dt.time

In [1135]:
df_final_instrumentos['Agente Representante'] = df_final_instrumentos['Agente Representante'].str.lower()

resumen_agentes = df_final_instrumentos.groupby('Agente Representante').agg(
    num_recursos=('Nombre Recurso', 'nunique'),
    recursos=('Nombre Recurso', lambda x: list(x.unique()))
).reset_index()

resumen_agentes

,Agente Representante,num_recursos,recursos
0,celsia colombia s.a. e.s.p.,11,"[ZIPAEMG 3, PROELECTRICA 1, CARTAGENA 2, GECEL..."
1,compañía eléctrica de sochagota s.a.s. e.s.p.,5,"[TESORITO, TERMOCANDELARIA CC, TERMOEMCALI CC,..."
2,empresas publicas de medellin e.s.p.,9,"[PAIPA 1, TERMOVALLE CC, MERILECTRICA 1, TERMO..."
3,enel colombia sa esp,13,"[TERMOYOPAL 1, GUAJIRA 2, BARRANQUILLA 4, BARR..."
4,enfragen termoflores s.a.s. e.s.p.,5,"[TASAJERO 2, TERMOYOPAL 2, TASAJERO 1, TERMOYO..."
5,enfragen termovalle s.a.s. empresa de servicio...,8,"[ZIPAEMG 2, ZIPAEMG 5, GECELCA 32, GECELCA 3, ..."
6,generadora termocentro s.a.s. e.s.p.,7,"[GUAJIRA 1, CARTAGENA 2, PROELECTRICA 1, GUAJI..."
7,generadora y comercializadora de energia del c...,21,"[TERMONORTE, TERMOSIERRA CC, TERMOCENTRO CC, T..."
8,gestion energetica s.a. e.s.p.,9,"[TERMODORADA 1, TEBSAB CC, TERMOCANDELARIA CC,..."
9,nitro energy colombia s.a.s. e.s.p.,9,"[TERMOYOPAL G4, TERMOYOPAL G3, TERMOYOPAL 1, T..."


In [1136]:
df_final_instrumentos[['CodigoPlanta', 'Nombre Recurso', 'Agente Representante']].value_counts()

CodigoPlanta  Nombre Recurso  Agente Representante                                           
ZPA5          BARRANQUILLA 4  enel colombia sa esp                                               66420
ZPA2          BARRANQUILLA 3  enel colombia sa esp                                               62856
GEC3          TERMONORTE      generadora y comercializadora de energia del caribe s.a. e.s.p.    60588
TSR1          TERMOVALLE CC   empresas publicas de medellin e.s.p.                               59868
PPA4          TESORITO        compañía eléctrica de sochagota  s.a.s. e.s.p.                     57888
                                                                                                 ...  
TVL1          TERMOYOPAL G4   enfragen termovalle s.a.s. empresa de servicios publicos             864
TSJ1          CARTAGENA 2     termotasajero s.a. e.s.p.                                            864
TFL1          ZIPAEMG 2       enfragen termoflores s.a.s. e.s.p.                  

## Bilaterales

In [1137]:
# archivo_contratos_bilaterales = "../datos/raw/Ventas_Contrato_(kwh)_2025.xlsx"
# df_bilaterales = pd.read_excel(archivo_contratos_bilaterales,header=3)


# archivo_agentes = "../datos/raw/Agentes.xlsx"
# df_agentes = pd.read_excel(archivo_agentes,header=4)
# df_agentes_sel = df_agentes[['CÓDIGO SIC', 'NOMBRE AGENTE']]

# df_desglose = pd.read_excel("../datos/raw/desglose.xlsx",header=3)

# # Hacer merge
# df_merged = pd.merge(
#     df_bilaterales,            # left DataFrame: todo
#     df_agentes_sel,            # right DataFrame: solo Código SIC y Nombre
#     how='left',                # left join: conserva todos de df_bilaterales
#     left_on='Código Agente',   # columna del left
#     right_on='CÓDIGO SIC'      # columna del right
# )

# # Opcional: eliminar la columna duplicada CÓDIGO SIC
# df_merged.drop(columns=['CÓDIGO SIC'], inplace=True)

# df_merged['NOMBRE AGENTE'] = df_merged['NOMBRE AGENTE'].str.lower()
# df_merged = df_merged[df_merged['NOMBRE AGENTE'].notna()].copy()
# df_merged[['Código Agente', 'NOMBRE AGENTE']].value_counts()
# plant_to_agent = {
#     "TBST": "TBSG",
#     "TBQ3": "TBSG",
#     "TBQ4": "TBSG",
#     "TRN1": "TMNG",
#     "TCDT": "TCIG",
#     "3ENA": "EPSG",
#     "MRL1": "EPSG",
#     "TVL1": "TMVG",
#     "ZPA2": "ENDG",
#     "ZPA3": "ENDG",
#     "ZPA4": "ENDG",
#     "ZPA5": "ENDG",
#     "TEC1": "TEMG",
#     "TDR1": "EPMG",
#     "TSR1": "EPMG",
#     "TYP1": "TYPG",
#     "TYP2": "TYPG",
#     "TYP3": "TYPG",
#     "TYP4": "TYPG",
#     "TFL1": "TMFG",
#     "TFL4": "TMFG",
#     "GEC3": "GECG",
#     "GE32": "GECG",
#     "TGJ1": "GECG",
#     "TGJ2": "GECG",
#     "PPA1": "HIMG",
#     "PPA2": "HIMG",
#     "PPA3": "HIMG",
#     "PPA4": "SOCG",
#     "CTG1": "NTCG",
#     "CTG2": "NTCG",
#     "PRG1": "PRIG",
#     "PRG2": "PRIG",
#     "TSJ1": "TRMG",
#     "TSJ2": "TERG",
#     "TRM1": "TCNG",
# }

# df_final_instrumentos["codigoAgente"] = df_final_instrumentos["CodigoPlanta"].map(plant_to_agent)

# df_final_instrumentos[['CodigoPlanta', 'Nombre Recurso', 'Agente Representante', 'codigoAgente']].value_counts()



In [1138]:
# df_merged_2 = df_merged.copy()
# df_final_2 = df_final_instrumentos.copy()

# df_merged_2["Código Agente"] = df_merged_2["Código Agente"].astype(str).str.strip().str.upper()
# df_final_2["codigoAgente"] = df_final_2["codigoAgente"].astype(str).str.strip().str.upper()

# df_merged_2["Fecha"] = pd.to_datetime(df_merged_2["Fecha"])
# df_final_2["Fecha"] = pd.to_datetime(df_final_2["Fecha"])

# cols_horas = [str(h) for h in range(24)]
# cols_merge = ["Fecha", "Código Agente", "Mercado"] + cols_horas

# df_final_instrumentos = pd.merge(
#     df_final_2,
#     df_merged_2[cols_merge],
#     left_on=["Fecha", "codigoAgente"],
#     right_on=["Fecha", "Código Agente"],
#     how="left"
# ).drop(columns=["Código Agente"])

In [1139]:
df_final_instrumentos["bilaterales"] = 0
df_final_instrumentos.loc[
    df_final_instrumentos["CodigoPlanta"].astype(str).str.strip().str.upper() == "PPA4",
    "bilaterales"
] = 130500

In [1140]:
df_final_instrumentos['Fecha'].unique()

<DatetimeArray>
['2025-05-25 00:00:00', '2025-04-07 00:00:00', '2025-07-01 00:00:00',
 '2025-07-02 00:00:00', '2025-04-08 00:00:00', '2025-08-15 00:00:00',
 '2025-09-03 00:00:00', '2025-08-28 00:00:00', '2025-06-12 00:00:00',
 '2025-06-09 00:00:00',
 ...
 '2025-05-14 00:00:00', '2025-08-25 00:00:00', '2025-06-24 00:00:00',
 '2025-04-10 00:00:00', '2025-05-15 00:00:00', '2025-05-01 00:00:00',
 '2025-04-09 00:00:00', '2025-06-17 00:00:00', '2025-08-12 00:00:00',
 '2025-09-05 00:00:00']
Length: 117, dtype: datetime64[us]

##  Estandarizacion

In [1141]:
cols_log = [
    "demanda_h00", "demanda_h01", "demanda_h02", "demanda_h03", "demanda_h04",
    "demanda_h05", "demanda_h06", "demanda_h07", "demanda_h08", "demanda_h09",
    "demanda_h10", "demanda_h11", "demanda_h12", "demanda_h13", "demanda_h14",
    "demanda_h15", "demanda_h16", "demanda_h17", "demanda_h18", "demanda_h19",
    "demanda_h20", "demanda_h21", "demanda_h22", "demanda_h23"
]

for c in cols_log:
    df_final_instrumentos[c] = pd.to_numeric(df_final_instrumentos[c], errors="coerce")
    df_final_instrumentos[c] = np.log1p(df_final_instrumentos[c])

In [1142]:
import numpy as np
import pandas as pd

cols_log = [
    "nivel_embalses", "aportes_rios",
    "demanda_h00", "demanda_h01", "demanda_h02", "demanda_h03", "demanda_h04",
    "demanda_h05", "demanda_h06", "demanda_h07", "demanda_h08", "demanda_h09",
    "demanda_h10", "demanda_h11", "demanda_h12", "demanda_h13", "demanda_h14",
    "demanda_h15", "demanda_h16", "demanda_h17", "demanda_h18", "demanda_h19",
    "demanda_h20", "demanda_h21", "demanda_h22", "demanda_h23"
]

# asegurar numéricas
for c in cols_log:
    df_final_instrumentos[c] = pd.to_numeric(df_final_instrumentos[c], errors="coerce")

# máscara única
mask = np.ones(len(df_final_instrumentos), dtype=bool)
for c in cols_log:
    mask &= np.isfinite(df_final_instrumentos[c]) & (df_final_instrumentos[c] > 0)

df_final_instrumentos = df_final_instrumentos.loc[mask].copy()

# aplicar log
for c in cols_log:
    df_final_instrumentos[c] = np.log(df_final_instrumentos[c])

In [1143]:
df_final_instrumentos[['EQpos', 'EdQb' ]]=df_final_instrumentos[['EQpos', 'EdQb']]
df_final_instrumentos["bilaterales"] = np.log1p(
    pd.to_numeric(df_final_instrumentos["bilaterales"], errors="coerce")
)
#df_final_instrumentos['precio']=df_final_instrumentos['precio']*1000

## Estimacion con lo de antes

In [1144]:
import numpy as np
import pandas as pd

PARAM_NAMES = [
    "phi_cons_i1", "phi_Pfuel_i1", "phi_Foil6_i1", "phi_TRM_i1", "phi_CERE_i1", "phi_FAZN_i1",
    "phi_cons_i2", "phi_Pfuel_i2", "phi_Foil6_i2", "phi_TRM_i2",
    "phi_cons_i3", "phi_Pfuel_i3", "phi_Foil6_i3", "phi_TRM_i3"
]

INSTRUMENT_COLS = [
    "nivel_embalses", "aportes_rios",
    "month", "month_4", "month_5", "month_6", "month_7", "month_8", "month_9",
    "day_Friday", "day_Monday", "day_Sunday", "day_Thursday", "day_Tuesday", "day_Wednesday",
    "pfuel","demanda_h00", "demanda_h01", "demanda_h02", "demanda_h03", "demanda_h04",
    "demanda_h05", "demanda_h06", "demanda_h07", "demanda_h08", "demanda_h09",
    "demanda_h10", "demanda_h11", "demanda_h12", "demanda_h13", "demanda_h14",
     "demanda_h15", "demanda_h16", "demanda_h17", "demanda_h18", "demanda_h19",
     "demanda_h20", "demanda_h21", "demanda_h22", "demanda_h23", 'prob_nino', 'bilaterales'
]

def prepare_daily_panel(df):
    df = df.copy()

    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df["Foil6"] = df["COMBUSTOLEO"]
    df["FAZN"] = df["FAZNI"]

    df["A_hour"] = df["EQpos"] + df["precio"] * df["EdQb"]
    df["D_hour"] = df["EdQb"]

    # Hora numérica
    if "Hora" in df.columns:
        hora_tmp = pd.to_datetime(df["Hora"].astype(str), format="%H:%M:%S", errors="coerce")
        df["Hora_num"] = hora_tmp.dt.hour
    elif "FechaHora" in df.columns:
        df["FechaHora"] = pd.to_datetime(df["FechaHora"], errors="coerce")
        df["Hora_num"] = df["FechaHora"].dt.hour
    else:
        raise ValueError("No existe 'Hora' ni 'FechaHora'.")

    # Q_iht y Qtilde_iht
    df = df.sort_values(["CodigoPlanta", "Fecha", "Hora_num"]).copy()
    df["Q_iht"] = pd.to_numeric(df["EQpos"], errors="coerce")
    

    df["Qtilde_iht"] = np.maximum(df["Q_iht"], 0.0)

    g_hour = df.groupby(["CodigoPlanta", "Fecha"], sort=False)["Qtilde_iht"]

    df["Q_prev"] = g_hour.shift(1)
    df["Q_prev"] = df["Q_prev"].fillna(df["Qtilde_iht"])

    df["Q_next"] = g_hour.shift(-1)
    df["Q_next"] = df["Q_next"].fillna(df["Qtilde_iht"])

    # Valor anterior (shift 1)
    # Supongamos que g es: g = df.groupby("Grupo")["Q_iht"]

    

    df["curv_hour"] = 2.0 * df["Q_iht"] - df["Q_prev"] - df["Q_next"]

    # S2: suma de Qtilde
    df["S2_hour"] = df["Qtilde_iht"]

    # S3: suma de (2Q - Qprev - Qnext)
    df["S3_hour"] = 2.0 * df["Q_iht"] - df["Q_prev"] - df["Q_next"]

    required_cols = [
        "CodigoPlanta", "Fecha",
        "pfuel", "Foil6", "trm", "CERE", "FAZN",
        "id_planta", "Combustible por Defecto", "Nombre Recurso", "Agente Representante"
    ] + INSTRUMENT_COLS

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas en el DataFrame: {missing}")

    agg_dict = {
        "A_it": ("A_hour", "sum"),
        "D_it": ("D_hour", "sum"),
        "S2_it": ("S2_hour", "sum"),
        "S3_it": ("S3_hour", "sum"),
        "pfuel": ("pfuel", "first"),
        "Foil6": ("Foil6", "first"),
        "trm": ("trm", "first"),
        "CERE": ("CERE", "first"),
        "FAZN": ("FAZN", "first"),
        "id_planta": ("id_planta", "first"),
        "combustible": ("Combustible por Defecto", "first"),
        "recurso": ("Nombre Recurso", "first"),
        "agente": ("Agente Representante", "first"),
    }

    for col in INSTRUMENT_COLS:
        agg_dict[col] = (col, "first")

    daily = (
        df.groupby(["CodigoPlanta", "Fecha"], as_index=False)
          .agg(**agg_dict)
    )

    num_cols = ["D_it", "S2_it", "S3_it", "pfuel", "Foil6", "trm", "CERE", "FAZN"] + INSTRUMENT_COLS
    for col in num_cols:
        daily[col] = pd.to_numeric(daily[col], errors="coerce")

    mask = np.isfinite(daily["D_it"]) & (daily["D_it"] != 0)

    for col in ["pfuel", "Foil6", "trm", "CERE", "FAZN", "S2_it", "S3_it"]:
        mask &= np.isfinite(daily[col])

    for col in INSTRUMENT_COLS:
        mask &= np.isfinite(daily[col])

    daily = daily.loc[mask].copy()

    daily["ln_pfuel"] = np.log(daily["pfuel"])
    daily["ln_Foil6"] = np.log(daily["Foil6"])
    daily["ln_TRM"]   = np.log(daily["trm"])
    daily["ln_CERE"]  = np.log(daily["CERE"])
    daily["ln_FAZN"]  = np.log(daily["FAZN"])

    return daily

In [1145]:
def build_X(sub):
    sub = sub.copy()

    s2 = sub["S2_it"].to_numpy(dtype=float)   # sum_h Qtilde_iht
    s3 = sub["S3_it"].to_numpy(dtype=float)   # sum_h (2Q_iht - Q_{h-1} - Q_{h+1})

    lp = sub["ln_pfuel"].to_numpy(dtype=float)
    lf = sub["ln_Foil6"].to_numpy(dtype=float)
    lt = sub["ln_TRM"].to_numpy(dtype=float)
    lc = sub["ln_CERE"].to_numpy(dtype=float)
    lz = sub["ln_FAZN"].to_numpy(dtype=float)

    X = np.column_stack([
        np.ones(len(sub)),   # phi_cons_i1
        lp,                  # phi_Pfuel_i1
        lf,                  # phi_Foil6_i1
        lt,                  # phi_TRM_i1
        lc,                  # phi_CERE_i1
        lz,                  # phi_FAZN_i1

        s2,                  # phi_cons_i2
        s2 * lp,             # phi_Pfuel_i2
        s2 * lf,             # phi_Foil6_i2
        s2 * lt,             # phi_TRM_i2

        s3,                  # phi_cons_i3
        s3 * lp,             # phi_Pfuel_i3
        s3 * lf,             # phi_Foil6_i3
        s3 * lt,             # phi_TRM_i3
    ]).astype(float)

    return X


# =========================================================
# 5. Construir Z_i (T x L) con tus instrumentos
# =========================================================
def build_Z_from_instrument_list(sub, add_constant=False, standardize=False):
    sub = sub.copy()

    Z = sub[INSTRUMENT_COLS].to_numpy(dtype=float)

    if standardize:
        mu = Z.mean(axis=0, keepdims=True)
        sd = Z.std(axis=0, ddof=0, keepdims=True)
        sd[sd == 0] = 1.0
        Z = (Z - mu) / sd

    if add_constant:
        Z = np.column_stack([np.ones(len(sub)), Z])

    return Z


# =========================================================
# 6. Estimación GMM por planta
# =========================================================
def estimate_one_plant_gmm(sub, W=None, add_constant_Z=False, standardize_Z=False):
    sub = sub.sort_values("Fecha").copy()

    A = sub["A_it"].to_numpy(dtype=float).reshape(-1, 1)
    D = np.diag(sub["D_it"].to_numpy(dtype=float))
    X = build_X(sub)
    Z = build_Z_from_instrument_list(
        sub,
        add_constant=add_constant_Z,
        standardize=standardize_Z
    )

    B = Z.T @ D @ X
    a = Z.T @ A

    L = Z.shape[1]
    K = X.shape[1]

    if W is None:
        W = np.eye(L)

    G = B.T @ W @ B
    rhs = B.T @ W @ a

    xi_hat = np.linalg.pinv(G) @ rhs
    xi_hat = xi_hat.flatten()

    out = {
        "n_obs": len(sub),
        "n_params": K,
        "n_instruments": L,
        "rank_X": np.linalg.matrix_rank(X),
        "rank_B": np.linalg.matrix_rank(B),
        "identified": np.linalg.matrix_rank(B) == K,
    }

    for name, val in zip(PARAM_NAMES, xi_hat):
        out[name] = val

    return out


# =========================================================
# 7. Estimar para todas las plantas
# =========================================================
def estimate_all_plants(df, W=None, min_obs=20, add_constant_Z=False, standardize_Z=False):
    daily = prepare_daily_panel(df)

    results = []
    for planta, sub in daily.groupby("CodigoPlanta", sort=True):
        sub = sub.copy()

        if len(sub) < min_obs:
            continue

        res = estimate_one_plant_gmm(
            sub,
            W=W,
            add_constant_Z=add_constant_Z,
            standardize_Z=standardize_Z
        )
        print(res)

        res["CodigoPlanta"] = planta
        res["id_planta"] = sub["id_planta"].iloc[0] if "id_planta" in sub.columns else np.nan
        res["Combustible por Defecto"] = sub["combustible"].iloc[0] if "combustible" in sub.columns else np.nan
        res["Nombre Recurso"] = sub["recurso"].iloc[0] if "recurso" in sub.columns else np.nan
        res["Agente Representante"] = sub["agente"].iloc[0] if "agente" in sub.columns else np.nan

        results.append(res)

    return pd.DataFrame(results)

In [1146]:
results_gmm = estimate_all_plants(
    df_final_instrumentos,
    W=None,
    min_obs=20,
    add_constant_Z=False,
    standardize_Z=True
)

{'n_obs': 117, 'n_params': 14, 'n_instruments': 42, 'rank_X': np.int64(9), 'rank_B': np.int64(8), 'identified': np.False_, 'phi_cons_i1': np.float64(4.752773186159247e-12), 'phi_Pfuel_i1': np.float64(2.644495326675967e-11), 'phi_Foil6_i1': np.float64(2.9231426836681294e-11), 'phi_TRM_i1': np.float64(3.655807068135282e-11), 'phi_CERE_i1': np.float64(1.819142307985319e-11), 'phi_FAZN_i1': np.float64(5.7623432513573405e-12), 'phi_cons_i2': np.float64(6.283814088625661e-05), 'phi_Pfuel_i2': np.float64(-1.6866956119021559e-06), 'phi_Foil6_i2': np.float64(-1.0903374302317914e-06), 'phi_TRM_i2': np.float64(-4.8493212236877525e-06), 'phi_cons_i3': np.float64(0.0), 'phi_Pfuel_i3': np.float64(0.0), 'phi_Foil6_i3': np.float64(0.0), 'phi_TRM_i3': np.float64(0.0)}
{'n_obs': 103, 'n_params': 14, 'n_instruments': 42, 'rank_X': np.int64(9), 'rank_B': np.int64(9), 'identified': np.False_, 'phi_cons_i1': np.float64(1.3880402540246764e-10), 'phi_Pfuel_i1': np.float64(9.729577742437212e-10), 'phi_Foil6_i1

In [1147]:
results_gmm

,n_obs,n_params,n_instruments,rank_X,rank_B,identified,phi_cons_i1,phi_Pfuel_i1,phi_Foil6_i1,phi_TRM_i1,...,phi_TRM_i2,phi_cons_i3,phi_Pfuel_i3,phi_Foil6_i3,phi_TRM_i3,CodigoPlanta,id_planta,Combustible por Defecto,Nombre Recurso,Agente Representante
0,117,14,42,9,8,False,4.752773e-12,2.644495e-11,2.923143e-11,3.655807e-11,...,-4.849321e-06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3ENA,29.0,GAS,GECELCA 32,celsia colombia s.a. e.s.p.
1,103,14,42,9,9,False,1.388040e-10,9.729578e-10,1.085517e-09,1.318818e-09,...,1.809414e-05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,CTG1,27.0,GAS,GUAJIRA 1,nitro energy colombia s.a.s. e.s.p.
2,101,14,42,9,9,False,5.616965e-11,3.836921e-10,4.327445e-10,5.242498e-10,...,7.623944e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,CTG2,28.0,GAS,TERMOYOPAL G4,nitro energy colombia s.a.s. e.s.p.
3,31,14,42,8,8,False,-2.474005e-13,-1.846979e-12,-2.406857e-12,-3.037631e-12,...,-9.651074e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,GE32,9.0,CARBON,CARTAGENA 2,generadora y comercializadora de energia del c...
4,111,14,42,9,9,False,-1.787056e-12,-7.872626e-12,-1.097424e-11,-1.307097e-11,...,-1.866999e-06,5.256281e-23,2.676122e-22,3.445201e-22,4.392882e-22,GEC3,13.0,CARBON,TERMONORTE,generadora y comercializadora de energia del c...
5,116,14,42,9,8,False,7.785036e-13,4.288910e-12,4.728695e-12,5.721994e-12,...,6.395229e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,MRL1,5.0,GAS,PROELECTRICA 1,celsia colombia s.a. e.s.p.
6,101,14,42,9,9,False,-3.858656e-11,-1.821278e-10,-2.468584e-10,-3.029284e-10,...,-2.127917e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,PPA1,6.0,CARBON,TERMODORADA 1,gestion energetica s.a. e.s.p.
7,102,14,42,9,9,False,4.798309e-12,3.565632e-11,5.022463e-11,5.728951e-11,...,1.205036e-06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,PPA2,11.0,CARBON,TERMOCANDELARIA CC,gestion energetica s.a. e.s.p.
8,72,14,42,9,9,False,-8.863970e-12,-3.619848e-11,-4.790697e-11,-5.989876e-11,...,1.160318e-06,-6.830643e-23,-3.473328e-22,-4.540655e-22,-5.712318e-22,PPA3,12.0,CARBON,TEBSAB CC,gestion energetica s.a. e.s.p.
9,106,14,42,9,9,False,8.277237e-13,3.924699e-12,5.280946e-12,6.407679e-12,...,-9.305123e-07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,PPA4,10.0,CARBON,TERMOEMCALI CC,compañía eléctrica de sochagota s.a.s. e.s.p.


## Computar costos marginales

In [1148]:
import numpy as np
import pandas as pd

PARAM_NAMES = [
    "phi_cons_i1", "phi_Pfuel_i1", "phi_Foil6_i1", "phi_TRM_i1", "phi_CERE_i1", "phi_FAZN_i1",
    "phi_cons_i2", "phi_Pfuel_i2", "phi_Foil6_i2", "phi_TRM_i2",
    "phi_cons_i3", "phi_Pfuel_i3", "phi_Foil6_i3", "phi_TRM_i3"
]

def add_phi_and_marginal_costs(df_base, results_gmm, q_col="EQpos"):
    df = df_base.copy()

    # --------------------------------------------------
    # 1. Fechas y hora
    # --------------------------------------------------
    if "FechaHora" in df.columns:
        df["FechaHora"] = pd.to_datetime(df["FechaHora"], errors="coerce")

    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce").dt.normalize()

    if "Hora" in df.columns:
        if np.issubdtype(df["Hora"].dtype, np.number):
            df["Hora_num"] = pd.to_numeric(df["Hora"], errors="coerce")
        else:
            hora_tmp = pd.to_datetime(df["Hora"].astype(str), format="%H:%M:%S", errors="coerce")
            df["Hora_num"] = hora_tmp.dt.hour
    elif "FechaHora" in df.columns:
        df["Hora_num"] = df["FechaHora"].dt.hour
    else:
        raise ValueError("No existe columna 'Hora' ni 'FechaHora'.")

    # --------------------------------------------------
    # 2. Variables necesarias
    # --------------------------------------------------
    if "Foil6" not in df.columns:
        if "COMBUSTOLEO" in df.columns:
            df["Foil6"] = df["COMBUSTOLEO"]
        else:
            raise ValueError("Falta 'Foil6' y también 'COMBUSTOLEO'.")

    if "FAZN" not in df.columns:
        if "FAZNI" in df.columns:
            df["FAZN"] = df["FAZNI"]
        else:
            raise ValueError("Falta 'FAZN' y también 'FAZNI'.")

    needed = ["pfuel", "Foil6", "trm", "CERE", "FAZN", q_col]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas en df_base: {missing}")

    for col in needed:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # --------------------------------------------------
    # 3. Logs de covariables
    # --------------------------------------------------
    df["ln_pfuel"] = np.where(df["pfuel"] > 0, np.log(df["pfuel"]), np.nan)
    df["ln_Foil6"] = np.where(df["Foil6"] > 0, np.log(df["Foil6"]), np.nan)
    df["ln_TRM"]   = np.where(df["trm"]   > 0, np.log(df["trm"]),   np.nan)
    df["ln_CERE"]  = np.where(df["CERE"]  > 0, np.log(df["CERE"]),  np.nan)
    df["ln_FAZN"]  = np.where(df["FAZN"]  > 0, np.log(df["FAZN"]),  np.nan)

    # --------------------------------------------------
    # 4. Construcción de Q_iht, Qtilde_iht, vecinos
    # --------------------------------------------------
    df = df.sort_values(["CodigoPlanta", "Fecha", "Hora_num"]).copy()

    df["Q_iht"] = df[q_col]
    df["Qtilde_iht"] = np.maximum(df["Q_iht"], 0.0)

    g = df.groupby(["CodigoPlanta", "Fecha"], sort=False)["Q_iht"]

    # hora anterior dentro del mismo día
    df["Q_prev"] = g.shift(1)
    df["Q_prev"] = df["Q_prev"].fillna(df["Qtilde_iht"])

    # hora siguiente dentro del mismo día
    df["Q_next"] = g.shift(-1)
    df["Q_next"] = df["Q_next"].fillna(df["Qtilde_iht"])

    df["Q_iht"] = df[q_col]
    df["Qtilde_iht"] = np.maximum(df["Q_iht"], 0.0)   # q_min = 0

    g = df.groupby(["CodigoPlanta", "Fecha"], sort=False)

    # Valor anterior (shift 1)
    # Supongamos que g es: g = df.groupby("Grupo")["Q_iht"]

    

    df["curv_hour"] = 2.0 * df["Q_iht"] - df["Q_prev"] - df["Q_next"]

    # --------------------------------------------------
    # 5. Agregados diarios correctos
    #    S2_it = sum_h Qtilde_iht
    #    S3_it = sum_h (2Q_iht - Q_prev - Q_next)
    # --------------------------------------------------
    daily_terms = (
        df.groupby(["CodigoPlanta", "Fecha"], as_index=False)
          .agg(
              S2_it=("Qtilde_iht", "sum"),
              S3_it=("curv_hour", "sum")
          )
    )

    df = df.merge(
        daily_terms,
        on=["CodigoPlanta", "Fecha"],
        how="left",
        validate="m:1"
    )

    # --------------------------------------------------
    # 6. Merge de parámetros por planta
    # --------------------------------------------------
    params_keep = ["CodigoPlanta"] + PARAM_NAMES
    missing_params = [c for c in params_keep if c not in results_gmm.columns]
    if missing_params:
        raise ValueError(f"Faltan columnas en results_gmm: {missing_params}")

    params = results_gmm[params_keep].drop_duplicates(subset=["CodigoPlanta"]).copy()

    df = df.merge(
        params,
        on="CodigoPlanta",
        how="left",
        validate="m:1"
    )

    # --------------------------------------------------
    # 7. Construir phi_it1, phi_it2, phi_it3
    # --------------------------------------------------
    df["phi_it1"] = (
        df["phi_cons_i1"]
        + df["phi_Pfuel_i1"] * df["ln_pfuel"]
        + df["phi_Foil6_i1"] * df["ln_Foil6"]
        + df["phi_TRM_i1"]   * df["ln_TRM"]
        + df["phi_CERE_i1"]  * df["ln_CERE"]
        + df["phi_FAZN_i1"]  * df["ln_FAZN"]
    )

    df["phi_it2"] = (
        df["phi_cons_i2"]
        + df["phi_Pfuel_i2"] * df["ln_pfuel"]
        + df["phi_Foil6_i2"] * df["ln_Foil6"]
        + df["phi_TRM_i2"]   * df["ln_TRM"]
    )

    df["phi_it3"] = (
        df["phi_cons_i3"]
        + df["phi_Pfuel_i3"] * df["ln_pfuel"]
        + df["phi_Foil6_i3"] * df["ln_Foil6"]
        + df["phi_TRM_i3"]   * df["ln_TRM"]
    )

    # --------------------------------------------------
    # 8. Costo marginal horario estimado
    # --------------------------------------------------
    df["mc_iht_hat"] = (
        df["phi_it1"]
        + df["phi_it2"] * df["Qtilde_iht"]
        + df["phi_it3"] * df["curv_hour"]
    )

    # --------------------------------------------------
    # 9. Costo marginal promedio diario estimado
    # --------------------------------------------------
    df["phi_bar_it"] = (
        df["phi_it1"]
        + df["phi_it2"] * df["S2_it"]
        + df["phi_it3"] * df["S3_it"]
    )

    return df

In [1149]:
df_final_instrumentos_costos[['CodigoPlanta', 'FechaHora','Q_iht', 'cantidad', 'Q_next', 'Q_prev', 'EQpos']]

,CodigoPlanta,FechaHora,Q_iht,cantidad,Q_next,Q_prev,EQpos
0,3ENA,2025-04-01 00:00:00,200000.0,200000.0,200000.0,200000.0,200000.0
1,3ENA,2025-04-01 00:00:00,200000.0,200000.0,200000.0,200000.0,200000.0
2,3ENA,2025-04-01 00:00:00,200000.0,200000.0,200000.0,200000.0,200000.0
3,3ENA,2025-04-01 00:00:00,200000.0,200000.0,200000.0,200000.0,200000.0
4,3ENA,2025-04-01 00:00:00,200000.0,200000.0,200000.0,200000.0,200000.0
...,...,...,...,...,...,...,...
3017263,ZPA5,2025-09-09 23:00:00,63000.0,63000.0,63000.0,63000.0,63000.0
3017264,ZPA5,2025-09-09 23:00:00,63000.0,63000.0,63000.0,63000.0,63000.0
3017265,ZPA5,2025-09-09 23:00:00,63000.0,63000.0,63000.0,63000.0,63000.0
3017266,ZPA5,2025-09-09 23:00:00,63000.0,63000.0,63000.0,63000.0,63000.0


In [1150]:
df_final_instrumentos_costos

,FechaHora,CodigoPlanta,precio,cantidad,cantidad_pos,Fecha,cluster,gamma_thumb,EQpos,EdQb,...,phi_TRM_i2,phi_cons_i3,phi_Pfuel_i3,phi_Foil6_i3,phi_TRM_i3,phi_it1,phi_it2,phi_it3,mc_iht_hat,phi_bar_it
0,2025-04-01 00:00:00,3ENA,889.28,200000.0,200000.0,2025-04-01,10,1.261930,200000.0,-4854.373719,...,-0.000005,0.0,0.0,0.0,0.0,7.519079e-10,0.000005,0.0,1.005028,868.344601
1,2025-04-01 00:00:00,3ENA,889.28,200000.0,200000.0,2025-04-01,10,1.261930,200000.0,-4854.373719,...,-0.000005,0.0,0.0,0.0,0.0,7.519079e-10,0.000005,0.0,1.005028,868.344601
2,2025-04-01 00:00:00,3ENA,889.28,200000.0,200000.0,2025-04-01,10,1.261930,200000.0,-4854.373719,...,-0.000005,0.0,0.0,0.0,0.0,7.519079e-10,0.000005,0.0,1.005028,868.344601
3,2025-04-01 00:00:00,3ENA,889.28,200000.0,200000.0,2025-04-01,10,1.261930,200000.0,-4854.373719,...,-0.000005,0.0,0.0,0.0,0.0,7.519079e-10,0.000005,0.0,1.005028,868.344601
4,2025-04-01 00:00:00,3ENA,889.28,200000.0,200000.0,2025-04-01,10,1.261930,200000.0,-4854.373719,...,-0.000005,0.0,0.0,0.0,0.0,7.519079e-10,0.000005,0.0,1.005028,868.344601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3017263,2025-09-09 23:00:00,ZPA5,338.94,63000.0,63000.0,2025-09-09,12,1.770429,63000.0,-673.709619,...,-0.000011,0.0,0.0,0.0,0.0,3.983113e-09,0.000006,0.0,0.396784,285.684628
3017264,2025-09-09 23:00:00,ZPA5,338.94,63000.0,63000.0,2025-09-09,12,1.770429,63000.0,-673.709619,...,-0.000011,0.0,0.0,0.0,0.0,3.983113e-09,0.000006,0.0,0.396784,285.684628
3017265,2025-09-09 23:00:00,ZPA5,338.94,63000.0,63000.0,2025-09-09,12,1.770429,63000.0,-673.709619,...,-0.000011,0.0,0.0,0.0,0.0,3.983113e-09,0.000006,0.0,0.396784,285.684628
3017266,2025-09-09 23:00:00,ZPA5,338.94,63000.0,63000.0,2025-09-09,12,1.770429,63000.0,-673.709619,...,-0.000011,0.0,0.0,0.0,0.0,3.983113e-09,0.000006,0.0,0.396784,285.684628


In [1151]:
df_final_instrumentos_costos = add_phi_and_marginal_costs(
    df_base=df_final_instrumentos,
    results_gmm=results_gmm,
    q_col="EQpos"
)

In [1152]:
costos_diarios = (
    df_final_instrumentos_costos[
        ["CodigoPlanta", "Fecha", "phi_it1", "phi_it2", "phi_it3", "S2_it", "S3_it", "phi_bar_it", 'precio','Combustible por Defecto']
    ]
    .drop_duplicates()
    .sort_values(["CodigoPlanta", "Fecha"])
    .reset_index(drop=True)
)

In [1153]:
costos_diarios

,CodigoPlanta,Fecha,phi_it1,phi_it2,phi_it3,S2_it,S3_it,phi_bar_it,precio,Combustible por Defecto
0,3ENA,2025-04-01,7.519079e-10,0.000005,0.0,172800000.0,0.0,868.344601,889.28,GAS
1,3ENA,2025-04-02,7.519584e-10,0.000005,0.0,172800000.0,0.0,867.787394,889.28,GAS
2,3ENA,2025-04-03,7.518875e-10,0.000005,0.0,172800000.0,0.0,868.901480,889.28,GAS
3,3ENA,2025-04-04,7.518014e-10,0.000005,0.0,172800000.0,0.0,871.539363,889.28,GAS
4,3ENA,2025-04-07,7.533485e-10,0.000005,0.0,172800000.0,0.0,839.610482,889.28,GAS
...,...,...,...,...,...,...,...,...,...,...
4061,ZPA5,2025-09-03,3.986948e-09,0.000006,0.0,45360000.0,0.0,276.508575,338.94,CARBON
4062,ZPA5,2025-09-04,3.986204e-09,0.000006,0.0,43432200.0,0.0,266.467297,338.94,CARBON
4063,ZPA5,2025-09-05,3.985551e-09,0.000006,0.0,47628000.0,0.0,293.792975,338.94,CARBON
4064,ZPA5,2025-09-08,3.983949e-09,0.000006,0.0,47628000.0,0.0,297.851190,338.94,CARBON


In [1154]:
costos_diarios.groupby('Combustible por Defecto')[['phi_it1', 'phi_it2', 'phi_it3', 'phi_bar_it']].mean()

,phi_it1,phi_it2,phi_it3,phi_bar_it
Combustible por Defecto,,,,
ACPM,2.325473e-07,0.000005,9.057827e-19,996.655015
CARBON,1.179624e-08,0.000005,3.695179e-20,267.124506
GAS,7.954469e-08,0.000017,3.222130e-19,866.277805
MEZCLA GAS - CARBÓN,4.133152e-09,0.000006,9.496709e-20,553.362107


In [1155]:
costos_diarios.groupby('Combustible por Defecto')[[ 'phi_bar_it', 'precio']].mean()

,phi_bar_it,precio
Combustible por Defecto,,
ACPM,996.655015,1089.006637
CARBON,267.124506,323.927581
GAS,866.277805,983.734109
MEZCLA GAS - CARBÓN,553.362107,534.860594


## Comparacion con los otros costos constantes

In [1156]:
df_input_red.columns

Index(['CodigoPlanta', 'available_q', 'phi', 'precio_d', 'daily_reman_demand',
       'Fecha', 'cluster', 'id_planta'],
      dtype='str')

In [1157]:
df_input_red['Fecha']=pd.to_datetime(df_input_red['Fecha'])
costos_diarios2=pd.merge(costos_diarios,df_input_red[['Fecha', 'CodigoPlanta', 'phi']], on=['Fecha', 'CodigoPlanta'], how='left')


In [1158]:
costos_diarios2=costos_diarios2.rename(columns={"phi_bar_it": "phi_ramping"})
costos_diarios2[['CodigoPlanta', 'Fecha', 'phi_ramping']]
costos_diarios2.to_csv('ramping.csv', index= False)

In [1167]:
df_input_red = pd.read_csv(
    "../results/fechas_117/TERMICA/inputRed2.csv",
    sep=";",
    decimal=","
)

df_input_red.columns = df_input_red.columns.str.strip()
costos_diarios2.columns = costos_diarios2.columns.str.strip()

df_input_red["Fecha"] = pd.to_datetime(df_input_red["Fecha"]).dt.normalize()
costos_diarios2["Fecha"] = pd.to_datetime(costos_diarios2["Fecha"]).dt.normalize()

df_input_red["CodigoPlanta"] = df_input_red["CodigoPlanta"].astype(str).str.strip()
costos_diarios2["CodigoPlanta"] = costos_diarios2["CodigoPlanta"].astype(str).str.strip()

# guardar orden original
cols_originales = df_input_red.columns.tolist()

phi_nuevo = (
    costos_diarios2[["CodigoPlanta", "Fecha", "phi_ramping"]]
    .drop_duplicates(subset=["CodigoPlanta", "Fecha"])
)

nuevo_input = df_input_red.merge(
    phi_nuevo,
    on=["CodigoPlanta", "Fecha"],
    how="left"
)

nuevo_input = nuevo_input.sort_values(["CodigoPlanta", "Fecha"])

# llenar faltantes con el valor del día anterior para la misma planta
nuevo_input["phi_nuevo"] = nuevo_input.groupby("CodigoPlanta")["phi_ramping"].ffill()

# reemplazar phi original
nuevo_input["phi"] = nuevo_input["phi_nuevo"].combine_first(nuevo_input["phi"])

# borrar auxiliares
nuevo_input = nuevo_input.drop(columns=["phi_ramping", "phi_nuevo"])

# restaurar exactamente el orden original de df_input_red
nuevo_input = nuevo_input[cols_originales]



In [ ]:
nuevo_input.to_csv(
    "../results/fechas_117/TERMICA/inputRedRamping.csv",
    sep=";",
    decimal=",",
    index=False,
    float_format="%.10f"   # evita notación científica y pérdida/artefactos
)

: 